In [1]:
import sqlite3
import requests
import time
import json
import pandas as pd
import urllib.parse
from dotenv import load_dotenv
import os
import datetime

In [2]:
load_dotenv()
ITAD_KEY = os.getenv('ITAD_API_KEY')

DB_PATH = 'gaming_warehouse.db'


In [3]:
#creo mi catallgo de identificadores únicos de ITAD y la relación entre nuestro catálogo principal y el de ITAD
def preparar_tablas_itad():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS CAT_ITAD_Juego (
            itad_id_texto TEXT PRIMARY KEY, -- El 'plain' de ITAD
            itad_titulo TEXT
        );
    """)
    
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS REL_Juego_ITAD (
            juego_id INTEGER,
            itad_id_texto TEXT,
            PRIMARY KEY (juego_id, itad_id_texto),
            FOREIGN KEY (juego_id) REFERENCES CAT_Juego(juego_id),
            FOREIGN KEY (itad_id_texto) REFERENCES CAT_ITAD_Juego(itad_id_texto)
        );
    """)
    
    conn.commit()
    conn.close()

In [4]:
preparar_tablas_itad()

In [6]:
def buscar_ids_itad(limite=50):
    conn = sqlite3.connect(DB_PATH)
    query = """
        SELECT j.juego_id, j.titulo 
        FROM CAT_Juego j
        LEFT JOIN REL_Juego_ITAD r ON j.juego_id = r.juego_id
        WHERE j.id_steam IS NOT NULL 
          AND r.itad_id_texto IS NULL
          AND j.categoria = 0
        LIMIT ?
    """
    juegos = conn.execute(query, (limite,)).fetchall()
    conn.close()

    if not juegos:
        print("No hay más juegos por vincular.")
        return

    print(f" {len(juegos)}  ")

    url_search = "https://api.isthereanydeal.com/games/search/v1"

    for juego_id, titulo in juegos:
        params = {
            "key": ITAD_KEY,
            "title": titulo,
            "limit": 1
        }
        
        try:
            res = requests.get(url_search, params=params, timeout=10)
            
            if res.status_code == 200:
                data = res.json()
                
                if data and len(data) > 0:
                    itad_id = data[0].get('id')
                    itad_slug = data[0].get('slug')
                    
                    if itad_id:
                        conn = sqlite3.connect(DB_PATH)
                        cursor = conn.cursor()
                        
                        cursor.execute("""
                            INSERT OR IGNORE INTO CAT_ITAD_Juego (itad_id_texto, itad_titulo) 
                            VALUES (?, ?)
                        """, (itad_id, titulo))
                        
                        cursor.execute("""
                            INSERT OR IGNORE INTO REL_Juego_ITAD (juego_id, itad_id_texto) 
                            VALUES (?, ?)
                        """, (juego_id, itad_id))
                        
                        conn.commit()
                        conn.close()
                        print(f"vinculado: {titulo} -> {itad_id}")
                    else:
                        print(f"sin ID en el JSON: {titulo}")
                else:
                    print(f"no se encontró en Search: {titulo}")
            
            elif res.status_code == 401:
                print("  ! Error 401 de api")
                break
            elif res.status_code == 429:
                print("limite")
                time.sleep(20)
                continue
                
        except Exception as e:
            print(f"Error con {titulo}: {e}")
        
        time.sleep(1)


In [7]:
#Prueba 5 de la funcion
buscar_ids_itad(limite=10)


No hay más juegos por vincular.


In [8]:
#ciclo para seguir buscando hasta que se acaben los juegos sin vincular
while True:
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("""
        SELECT COUNT(*) 
        FROM CAT_Juego j
        LEFT JOIN REL_Juego_ITAD r ON j.juego_id = r.juego_id
        WHERE j.id_steam IS NOT NULL 
          AND r.itad_id_texto IS NULL
          AND j.categoria = 0
    """)
    pendientes = cursor.fetchone()[0]
    conn.close()

    if pendientes == 0:
        print("Se acabaron los juegos a procesar")
        break

    print(f"\nFaltan {pendientes} juegos por vincular")
    
    buscar_ids_itad(limite=100)
    
    print("pausa")
    time.sleep(30)

Se acabaron los juegos a procesar


In [9]:
def Crear_tablas_ITAD_precio():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS CAT_Tienda (
            id_tienda INTEGER PRIMARY KEY,
            nombre TEXT
        )
    """)
    
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS Hist_Precios_ITAD (
            itad_id_texto TEXT,
            id_tienda INTEGER,
            precio REAL,
            corte_descuento INTEGER,
            fecha_unix INTEGER,
            PRIMARY KEY (itad_id_texto, id_tienda, fecha_unix)
        )
    """)

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS Datos_Actuales_ITAD (
            itad_id_texto TEXT PRIMARY KEY,
            precio_actual REAL,
            tienda_actual INTEGER,
            precio_minimo REAL,
            tienda_minimo INTEGER,
            fecha_minimo INTEGER,
            en_bundle INTEGER DEFAULT 0
        )
    """)
    conn.commit()
    conn.close()

In [10]:
Crear_tablas_ITAD_precio()

In [11]:
def procesar_lote_itad(meses_atras=1, limite_juegos=10):
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    
    cursor.execute("""
        SELECT r.itad_id_texto 
        FROM REL_Juego_ITAD r
        LEFT JOIN Datos_Actuales_ITAD d ON r.itad_id_texto = d.itad_id_texto
        WHERE d.itad_id_texto IS NULL
        LIMIT ?
    """, (limite_juegos,))
    juegos = cursor.fetchall()
    
    if not juegos:
        print("Sin juegos pendientes.")
        conn.close()
        return False

    for (itad_id,) in juegos:
        print(f"[{itad_id}]")
        body = [itad_id]
        
        cursor.execute("SELECT MAX(fecha_unix) FROM Hist_Precios_ITAD WHERE itad_id_texto = ?", (itad_id,))
        ultimo = cursor.fetchone()[0]
        
        if ultimo:
            fecha_dt = datetime.datetime.fromtimestamp(ultimo, datetime.timezone.utc)
        else:
            fecha_dt = datetime.datetime.now(datetime.timezone.utc) - datetime.timedelta(days=meses_atras * 30)
        
        since_iso = fecha_dt.strftime('%Y-%m-%dT%H:%M:%SZ')

        try:
            res_hist = requests.get("https://api.isthereanydeal.com/games/history/v2", params={
                "key": ITAD_KEY, "id": itad_id, "country": "MX", "since": since_iso
            })
            nuevos = 0
            if res_hist.status_code == 200:
                for cambio in res_hist.json():
                    t_id    = cambio.get('shop', {}).get('id')
                    t_name  = cambio.get('shop', {}).get('name', 'Desconocida')
                    deal    = cambio.get('deal', {})
                    precio  = deal.get('price', {}).get('amount')
                    cut     = deal.get('cut', 0)
                    fecha_str = cambio.get('timestamp')
                    
                    if t_id and precio and fecha_str:
                        ts = int(datetime.datetime.fromisoformat(fecha_str).timestamp())
                        cursor.execute("INSERT OR IGNORE INTO CAT_Tienda (id_tienda, nombre) VALUES (?, ?)", (t_id, t_name))
                        cursor.execute("""
                            INSERT OR IGNORE INTO Hist_Precios_ITAD (itad_id_texto, id_tienda, precio, corte_descuento, fecha_unix)
                            VALUES (?, ?, ?, ?, ?)
                        """, (itad_id, t_id, precio, cut, ts))
                        if cursor.rowcount > 0:
                            nuevos += 1
            print(f"  historial: {nuevos} nuevos")

            precio_min = tienda_min = fecha_min = None
            res_low = requests.post(f"https://api.isthereanydeal.com/games/historylow/v1?key={ITAD_KEY}&country=MX", json=body)
            if res_low.status_code == 200 and res_low.json():
                low = res_low.json()[0].get('low')
                if low:
                    precio_min = low.get('price', {}).get('amount')
                    tienda_min = low.get('shop', {}).get('id')
                    if low.get('timestamp'):
                        fecha_min = int(datetime.datetime.fromisoformat(low['timestamp']).timestamp())

            precio_act = tienda_act = None
            res_prices = requests.post(f"https://api.isthereanydeal.com/games/prices/v2?key={ITAD_KEY}&country=MX", json=body)
            if res_prices.status_code == 200 and res_prices.json():
                deals = res_prices.json()[0].get('deals')
                if deals:
                    precio_act = deals[0]['price']['amount']
                    tienda_act = deals[0]['shop']['id']

            en_bundle = 0
            res_b = requests.post(f"https://api.isthereanydeal.com/games/bundles/v2?key={ITAD_KEY}", json=body)
            if res_b.status_code == 200 and res_b.json():
                en_bundle = 1 if res_b.json()[0].get('bundles') else 0

            cursor.execute("""
                INSERT OR REPLACE INTO Datos_Actuales_ITAD
                (itad_id_texto, precio_actual, tienda_actual, precio_minimo, tienda_minimo, fecha_minimo, en_bundle)
                VALUES (?, ?, ?, ?, ?, ?, ?)
            """, (itad_id, precio_act, tienda_act, precio_min, tienda_min, fecha_min, en_bundle))
            conn.commit()

        except Exception as e:
            print(f"  error: {e}")

        time.sleep(1)

    conn.close()
    return True

In [12]:
procesar_lote_itad(meses_atras=1, limite_juegos=5)

[018d937f-4a27-72c8-9837-8ec5e5e92932]
  historial: 8 nuevos
[018d937f-6bca-706d-9e69-8316170c379b]
  historial: 55 nuevos
[018e147e-9b32-7025-90af-981a785efddf]
  historial: 54 nuevos
[018dd0ec-ce78-7205-887b-cfc7b259ae00]
  historial: 12 nuevos
[018d937f-30fa-705e-8a3a-f39719bdde93]
  historial: 5 nuevos


True

In [ ]:
while True:
    continuar = procesar_lote_itad(meses_atras=2, limite_juegos=200)
    if not continuar:
        break
    print("Espera")
    time.sleep(10)

[018d937f-131c-708f-a973-4fc0767e382d]
  historial: 50 nuevos
[018d937f-2979-72d3-b30e-beb459582423]
  historial: 11 nuevos
[018d937f-060b-7399-9cd9-f894b77fb9a9]
  historial: 5 nuevos
[018d937f-00e8-729d-a917-e2f1d5bcc95d]
  historial: 91 nuevos
[018d937f-1d10-738e-a02f-67185816bae5]
  historial: 18 nuevos
[018d937f-1ec4-7327-8a2a-99c60e52cd77]
  historial: 6 nuevos
[018d937e-f54c-7379-a57f-9ab6cdc42d20]
  historial: 69 nuevos
[018d937f-0184-7248-8d64-3c723c523111]
  historial: 49 nuevos
[018d937f-206b-7116-a860-da40b43874db]
  historial: 6 nuevos
[018d937f-4614-7311-8f5e-54720a2e73c2]
  historial: 55 nuevos
[018d937e-fde4-72ff-a7af-45e4955a8dd6]
  historial: 0 nuevos
[018d937e-fdb9-7203-a89a-1be9ff5e9467]
  historial: 3 nuevos
[018d937e-ffba-7200-8bc4-99eccd424fa1]
  historial: 11 nuevos
[018d937f-2ee6-704c-8a25-f9606ccdf59a]
  historial: 7 nuevos
[018d937f-5c96-7241-a70f-e8de1c1e0609]
  historial: 118 nuevos
[018d937e-ffb9-707b-a6dd-86c93c60b01f]
  historial: 82 nuevos
[018d937e-ff8

In [11]:
#cuantows juegos sin vincular me quedan
conn = sqlite3.connect(DB_PATH)

query = """
SELECT COUNT(*) 
        FROM CAT_Juego j
        LEFT JOIN REL_Juego_ITAD r ON j.juego_id = r.juego_id
        WHERE j.id_steam IS NOT NULL 
          AND r.itad_id_texto IS NULL
          AND j.categoria = 0
"""

df_status = pd.read_sql_query(query, conn)
conn.close()

display(df_status)

,COUNT(*)
0,0


In [ ]:
# import sqlite3
# try:
#     conn = sqlite3.connect(DB_PATH)
#     conn.close()
#     print("Candado liberado.")
# except:
#     print("no se libero")

Candado liberado.
